# Lab 1: Standard RAG (Retrieval-Augmented Generation)

**학습 목표**
- RAG의 기본 파이프라인 이해: 청킹 → 임베딩 → 검색 → 생성
- FAISS 벡터 인덱스의 동작 원리 파악
- 청크 크기와 검색 결과의 관계 실험

**소요 시간**: 30-40분  
**필요 API**: OpenAI API Key

In [ ]:
import sys
sys.path.insert(0, '..')

from rag_lab import EmbeddingEngine, StandardRAG, load_papers
from rag_lab.utils import chunk_text

## 1. 논문 로딩

5편의 NBER Working Paper를 로드합니다.

In [ ]:
papers = load_papers()

for name, text in papers.items():
    print(f"{name}: {len(text):,} 글자 ({len(text)//4:,} 토큰 추정)")

## 2. 청킹 (Chunking) 실험

문서를 작은 조각(청크)으로 나눕니다.

**핵심 질문**: 청크 크기가 검색 품질에 어떤 영향을 미칠까?

In [ ]:
# 첫 번째 논문으로 청킹 실험
sample = list(papers.values())[0]

# 다양한 청크 크기 비교
for size in [500, 1000, 2000, 4000]:
    chunks = chunk_text(sample, chunk_size=size, overlap=size//10)
    avg_len = sum(len(c) for c in chunks) / len(chunks)
    print(f"청크 크기={size:>5}: {len(chunks):>3}개 청크, 평균 {avg_len:.0f} 글자")

In [ ]:
# 청크 내용 살펴보기
chunks = chunk_text(sample, chunk_size=2000, overlap=200)

print("=== 첫 번째 청크 ===")
print(chunks[0][:500])
print("\n=== 마지막 청크 ===")
print(chunks[-1][:500])

### 🤔 생각해보기
- 청크가 너무 작으면? → 문맥을 잃음 ("이 결과는..."에서 "이"가 무엇인지 모름)
- 청크가 너무 크면? → 검색 정밀도 떨어짐 (관련없는 내용이 섞임)
- overlap은 왜 필요한가? → 청크 경계에서 잘린 문장 복구

## 3. 임베딩 & FAISS 인덱스

In [ ]:
# 임베딩 엔진 로딩 (MiniLM-L6-v2, 384차원)
engine = EmbeddingEngine()

# 텍스트를 벡터로 변환
test_texts = [
    "intergenerational mobility and human capital",
    "negative interest rates and bank performance",
    "online auction market efficiency",
]
vectors = engine.encode(test_texts)
print(f"입력: {len(test_texts)}개 텍스트")
print(f"출력: {vectors.shape} (문장 수 × 차원 수)")

In [ ]:
# 코사인 유사도 직접 계산해보기
import numpy as np

# 정규화된 벡터의 내적 = 코사인 유사도
similarity_matrix = vectors @ vectors.T

print("유사도 행렬:")
for i, t1 in enumerate(test_texts):
    for j, t2 in enumerate(test_texts):
        print(f"  {t1[:30]:>30} vs {t2[:30]:<30} = {similarity_matrix[i][j]:.3f}")

## 4. Standard RAG 구축 & 질의

In [ ]:
# RAG 시스템 구축
rag = StandardRAG(papers, engine)

print(f"총 청크 수: {len(rag.chunks)}")
print(f"FAISS 인덱스 크기: {rag.index.ntotal} 벡터")

In [ ]:
# 검색만 먼저 해보기 (LLM 호출 없이)
question = "What is the effect of negative interest rates on banks?"

retrieved = rag.retrieve_only(question, top_k=3)
for i, r in enumerate(retrieved):
    print(f"\n--- 결과 #{i+1} (유사도: {r['score']:.3f}, 출처: {r['source']}) ---")
    print(r['chunk'][:300] + "...")

In [ ]:
# 전체 RAG 파이프라인 실행 (검색 + LLM 생성)
result = rag.query("What is the main finding about intergenerational mobility?")

print(f"검색 시간: {result.retrieval_time:.2f}초")
print(f"생성 시간: {result.generation_time:.2f}초")
print(f"총 시간:   {result.total_time:.2f}초")
print(f"출처: {result.sources}")
print(f"\n답변:\n{result.answer}")

## 5. 실험: 청크 크기 변경

**과제**: 청크 크기를 500, 1000, 4000으로 바꿔서 RAG를 구축하고, 같은 질문에 대한 답변 품질을 비교하세요.

In [ ]:
# TODO: 학생 실습
# rag_small = StandardRAG(papers, engine, chunk_size=500, chunk_overlap=50)
# rag_large = StandardRAG(papers, engine, chunk_size=4000, chunk_overlap=400)
# 
# question = "Compare the methodological approaches across papers"
# result_small = rag_small.query(question)
# result_large = rag_large.query(question)
# 
# print("=== 작은 청크 ===")
# print(result_small.answer[:300])
# print("\n=== 큰 청크 ===")
# print(result_large.answer[:300])

## 6. Standard RAG의 한계

다음 질문으로 Standard RAG의 한계를 확인해보세요:

```python
# 크로스 논문 질문 — RAG가 약한 영역
result = rag.query(
    "How do the information asymmetry themes in the credit check "
    "paper relate to auction market dynamics?"
)
```

**관찰 포인트**:
- 검색된 청크가 각각 다른 논문에서 왔는가?
- LLM이 청크 간의 연결을 잘 만들어내는가?
- 다음 Lab의 LightRAG, Wiki와 비교할 준비!